In [ ]:
import os, sys, json
from pathlib import Path
from types import SimpleNamespace
runners = list(Path("/kaggle/input").rglob("phase5_b1_train.py"))
print("Mounted runners:", [str(p) for p in runners], flush=True)
if len(runners) != 1: raise RuntimeError("Expected one official-data runner")
input_dir = str(runners[0].parent)
sys.path[:0] = [input_dir + "/phase1_source", input_dir]
import phase5_b1_train as runner
"""LoRMA+ output-side multiplicative adaptation from Bihany et al. (ACL 2025).

For a frozen linear weight W0, computes (I + alpha/rank * B A) W0 x.
The zero B initialization preserves the exact Phase 1 function at step zero.
"""
import torch
from torch import nn
from torch.nn import functional as F


class LoRMAPlusLinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int = 8, alpha: float = 16):
        super().__init__()
        if rank < 1 or base.bias is not None:
            raise ValueError("Expected positive rank and bias-free base linear")
        self.base = base
        self.rank = rank
        self.scale = alpha / rank
        self.a = nn.Parameter(torch.empty(rank, base.out_features))
        self.b = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.a, a=5**.5)

    def forward(self, x):
        frozen_output = self.base(x)
        transformed = F.linear(F.linear(frozen_output, self.a), self.b)
        return frozen_output + transformed * self.scale

runner.LoRALinear = LoRMAPlusLinear
output_dir = Path("/kaggle/working/lorma_plus")
args = SimpleNamespace(input=input_dir, output=str(output_dir), epochs=2, batch=8, eval_batch=16, accum=2, lr=0.0002)
runner.train(args)
report_file = output_dir / "phase5_b1_lora_report.json"
report = json.loads(report_file.read_text())
report["method"] = "LoRMA+ multiplicative output-side adapter"
report["method_reference"] = "https://aclanthology.org/2025.findings-acl.527/"
(output_dir / "phase5_b1_lorma_report.json").write_text(json.dumps(report, indent=2))
(output_dir / "phase5_b1_lora_prediction.zip").rename(output_dir / "phase5_b1_lorma_prediction.zip")
